In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# from src.add_fed_pca import process_clients_with_grouped_pca_rmse
from src.add_fed_pca import process_clients_with_grouped_pca_rmse

processed_dir = "datasets\\exp4\\gdlc"

with open(os.path.join(processed_dir, "gdlc_features.json"), "r") as f:
    gdlc_features = json.load(f)

original_dfs = {}
for name, df_dict in gdlc_features.items():
    original_dfs[name] = pd.read_parquet(f"{os.path.join(processed_dir, name)}.parquet")[df_dict['gdlc_features']]

In [2]:
def compute_rmse(original: pd.DataFrame, reconstructed: pd.DataFrame) -> float:
    diff = original.values - reconstructed.values
    mse = np.mean(diff ** 2)
    return np.sqrt(mse)

def local_PCA(original_df, n_components):
    pca_local = PCA(n_components=n_components)
    X_r_local = pca_local.fit_transform(original_df)
    local_df = pd.DataFrame(pca_local.inverse_transform(X_r_local), columns=original_df.columns)
    rmse_local = compute_rmse(original_df, local_df)
    return rmse_local

In [3]:
# rmse_local = {}
# for k in range(1, 11):
#     rmse_list = []
#     for df in original_dfs.values():
#         rmse_list.append(local_PCA(df, k))
#     rmse_local[k] = float(np.mean(rmse_list))
# rmse_local

In [4]:
rmse_global = {}
for k in range(1, 11):
    print(f"==>> k: {k}")
    fed_pca_dfs_dict, errors, pca_columns = process_clients_with_grouped_pca_rmse(
        dfs_dict=original_dfs,
        n_components=k
    )
    rmse_global[k] = errors
    # rmse_global[k] = float(np.mean(rmse_list))
# rmse_global

==>> k: 1


INFO:src.add_fed_pca:Computing local PCA for client client_0
INFO:src.add_fed_pca:Computing local PCA for client client_1
INFO:src.add_fed_pca:Computing local PCA for client client_2
INFO:src.add_fed_pca:Computing local PCA for client client_3
INFO:src.add_fed_pca:Computing local PCA for client client_4
INFO:src.add_fed_pca:Computing local PCA for client test


In [ ]:
avg_rmse_local = [float(np.mean(list(errors_dicts['reconstruction_errors_local'].values()))) for errors_dicts in list(rmse_global.values())]
avg_rmse_local

In [ ]:
avg_rmse_global = [float(np.mean(list(errors_dicts['reconstruction_errors_federated'].values()))) for errors_dicts in list(rmse_global.values())]
avg_rmse_global

In [ ]:
results = []
for k in range(1, 11):
    results.append({'method': 'FedPCA', 'n_components': k, 'rmse': avg_rmse_global[k-1]})
    results.append({'method': 'Local PCA', 'n_components': k, 'rmse': avg_rmse_local[k-1]})
df_rmse = pd.DataFrame(results)
print(df_rmse)

In [ ]:
method = 'FedPCA'
subset = df_rmse[df_rmse['method'] == 'FedPCA']

# plot it
plt.figure()
plt.plot(subset['n_components'], subset['rmse'], 'o-', label=method)
plt.xlabel('Number of PCA Components')
plt.ylabel('Reconstruction RMSE')
plt.title('FedPCA vs Local PCA Reconstruction Error (1–10 Components)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
for method, style in zip(['FedPCA', 'Local PCA'], ['o-', 's--']):
    subset = df_rmse[df_rmse['method'] == method]
    plt.plot(subset['n_components'], subset['rmse'], style, label=method)


plt.xlabel('Number of PCA Components')
plt.ylabel('Reconstruction RMSE')
plt.title('FedPCA vs Local PCA Reconstruction Error (1–10 Components)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

name_map = {
    'cic_bot_iot_1m':   'CIC-Bot-IoT',
    'cic_ton_iot_1m':   'CIC-ToN-IoT',
    'cic_ids_2017_1m':  'CIC-IDS-2017',
    'cic_unsw_1m':      'CIC-UNSW-NB15',
    'cic_ddos_2019_1m': 'CIC-DDoS-2019',
    'test':             'Test Set'
}

In [ ]:
n_comp = 3
reconstruction_errors_local = rmse_global[n_comp]['reconstruction_errors_local']
reconstruction_errors_federated = rmse_global[n_comp]['reconstruction_errors_federated']

clients = list(reconstruction_errors_local.keys())

local_vals = [reconstruction_errors_local[c] for c in clients]
fed_vals = [reconstruction_errors_federated[c] for c in clients]

x = np.arange(len(clients))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, fed_vals, width, label='FedPCA RMSE')
plt.bar(x + width/2, local_vals, width, label='Local PCA RMSE')

datasets = [gdlc_features[c]['dataset_name'] for c in clients]
datasets_names = [name_map[k] for k in datasets]
plt.xticks(x, datasets_names)
plt.xlabel('Client')
plt.ylabel('Reconstruction RMSE')
plt.title('Local vs. Federated Reconstruction Error per Client')
plt.legend()
plt.tight_layout()
plt.show()
